# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koushalkarthik15/mlflyrankkarthik/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** How can we accurately predict which high-traffic webpages are actively decaying in Google Search performance?

**Decision Supported:** This supports the content triage decision—identifying which pages editorial teams should prioritize for proactive refreshes before traffic is permanently lost.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Data:** FlyRank ML Internship starter dataset (`content_refresh_anonymized.csv`), encompassing March 2026.
**Exclusions:** `trend_pct` and `trend_direction` were strictly excluded from the feature set to prevent massive label leakage.
**Safety:** `client_id` and `content_id` were used exclusively for splitting data. No client PII or exact URLs are used.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
features = ['content_age_days', 'impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'word_count']
df[features] = df[features].fillna(0)

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Method:** Random Forest Classifier (max_depth=5).
**Features:** Age, impressions, clicks, position, CTR, word count.
**Label:** `is_declining` (derived from `trend_direction == down`).
**Baseline:** Rule-based logic `(age >= 180) * (impressions >= 1000) * impressions`.
**Validation:** GroupShuffleSplit by `client_id` (80/20) to ensure generalization to unseen clients.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(train[features], train['is_declining'])

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The Random Forest significantly outperformed both Random Guessing and our rigid Baseline approach.

In [ ]:
test['model_score'] = model.predict_proba(test[features])[:, 1]
test['baseline_score'] = (test['content_age_days'] >= 180).astype(int) * (test['impressions_90d'] >= 1000).astype(int) * test['impressions_90d']

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(f"Base Rate (Random guessing): {test['is_declining'].mean():.3f}")
print(f"Baseline Precision@50:       {precision_at_k(test['baseline_score'], test['is_declining'], k=50):.3f}")
print(f"Model Precision@50:          {precision_at_k(test['model_score'], test['is_declining'], k=50):.3f}")

## 5. Limitations

*What this work cannot claim.*

This model is a directional decision-support tool. It flags observed risk based on historical patterns, but it cannot differentiate between actively decaying content and stable "evergreen" reference documents. Human review is required.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Content teams should pull the Top 100 highest-scored pages weekly, filter out corporate/evergreen references, and manually review the remainder for necessary rewrites or metadata updates.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
plt.figure(figsize=(8, 5))
importances.plot(kind='barh', color='skyblue')
plt.title('Random Forest Feature Importances')
plt.xlabel('Importance')
os.makedirs('../figures', exist_ok=True)
plt.savefig('../figures/feature_importances.png', bbox_inches='tight')
plt.show()

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.